# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR$^2$ dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/api/python/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset via the Croissant schema URL
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object
meta = dataset.metadata

print(f"{meta.name}: {meta.description}")
print(f"\nMetadata Identifier (@id): {meta.id}")
print(f"Version: {meta.version}  |  License: {meta.license}\n")

## 2. Data Overview
Let's list available record sets with their `@id`. We'll preview some fields and their IDs from the first record set.

In [ ]:
# Get all record sets in the dataset
record_sets = list(dataset.record_sets)

if not record_sets:
    print("No record sets found in the dataset metadata.")
else:
    print(f"Found {len(record_sets)} record sets:")
    for rs in record_sets:
        print(f"- {rs.id}  (name: {getattr(rs, 'name', '')})")
    # Display field and column ids for first record set
    first_rs = record_sets[0]
    print(f"\nExample fields/columns in record set '{first_rs.id}':")
    for field in getattr(first_rs, 'fields', []):
        print(f"  Field: {field.id} (name: {getattr(field, 'name', '')})")
        for column in getattr(field, 'columns', []):
            print(f"    Column: {column.id} (name: {getattr(column, 'name', '')})")

## 3. Data Extraction
Load data from all available record sets into pandas DataFrames. Data extraction is performed by referencing record set, field, and column `@id`s.

In [ ]:
# Prepare a dictionary of DataFrames for each record set by @id

dataframes = {}

if not record_sets:
    print("Cannot extract records, no record sets available.")
else:
    for rs in record_sets:
        rs_id = rs.id
        print(f"Extracting records for record set: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records with columns: {df.columns.tolist()}")
        else:
            print(f"No records found for record set: {rs_id}")
    # Show head for the first available DataFrame
    if dataframes:
        first_rs_id = list(dataframes.keys())[0]
        print(f"\nFirst 5 rows for record set '@id': {first_rs_id}")
        display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply standard data processing steps such as filtering, normalization, and grouping. All references use Croissant `@id` fields.

In [ ]:
# Example EDA: If available, select a numeric field from the first record set

if not dataframes:
    print("No data loaded to perform EDA.")
else:
    # We'll use the first DataFrame
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id]
    print(f"Analyzing DataFrame for record set '@id': {rs_id}")

    # Try to infer a numeric field by looking for a numeric dtype or common column names
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_candidates:
        # Fallback: try to find plausible column names
        for col in df.columns:
            if any(keyword in col.lower() for keyword in ['log', 'score', 'value', 'coef', 'error', 'prob', 'age', 'count']):
                try:
                    df[col] = pd.to_numeric(df[col])
                    if not df[col].isnull().all():
                        numeric_candidates.append(col)
                except:
                    continue

    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]  # Use the first found numeric field's @id
        print(f"\nUsing numeric field for filtering and normalization: {numeric_field_id}")

        # Filter: select records with value above a chosen threshold
        threshold = df[numeric_field_id].quantile(0.75)
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize (z-score)
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try grouping by a categorical field (pick if available and has few categories)
        group_field = None
        for col in df.columns:
            if df[col].dtype == 'object' and df[col].nunique() > 1 and df[col].nunique() < 10:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame(f"mean_{numeric_field_id}")
            print(f"\nGrouped average of {numeric_field_id} by '{group_field}':")
            display(grouped_df)
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric field found for EDA.")

## 5. Visualization
Visualize distributions or relationships in the data using `matplotlib` and `seaborn`.

In [ ]:
# Visualization of the numeric field distribution and its relationship with a categorical field
if not dataframes:
    print("No data available for visualization.")
elif not numeric_candidates:
    print("No numeric fields to visualize.")
else:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If a categorical field was found, plot boxplot
    if group_field:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field)
        plt.show()

## 6. Conclusion
In this notebook, you've seen how to:
- Load a Croissant-based dataset package using the `mlcroissant` library.
- Discover and access record sets, fields, and columns by their `@id` identifiers.
- Extract record sets directly into pandas DataFrames for analysis.
- Perform filtering, normalization, and group-wise exploration, strictly using references by `@id`.
- Visualize the outcomes with common Python plotting libraries.

For further analysis, you can repeat similar steps for other record sets or fields, guided by their `@id` and the Croissant schema.